# Part 2 · Notebook 03 — Return distributions and stylized facts

**Sessions:** S3 (Return distributions & stylized facts) · Clinic W1 · [Lesson plan](../../docs/lessons/PART_02_QUANT_TOOLKIT.md)

**You will:**
1. Compute log returns and see why they add up over time.
2. Measure fat tails and skew.
3. Verify the stylized facts of returns on 10 tickers (clinic deliverable).

How these notebooks work: the loading and plotting code is written for you. Cells marked **✍️ Your turn** need 1–5 lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p2lib.py is in notebooks/part02/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p2lib as p

p.use_course_style()
pd.set_option("display.float_format", "{:,.4f}".format)
prices = p.load_prices()          # dates × 10 tickers (course data via P2_DATA, else synthetic)
rets = p.log_returns(prices)      # daily log returns
prices.tail(3)

## 1. Log returns

$r_t = \ln(P_t / P_{t-1})$

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
# ✍️ daily log returns of `prices` (drop the first empty row)
my_rets = ...
my_rets = p.check("log returns", my_rets, p.log_returns(prices))

In [ ]:
# ✍️ total simple return of SPY over the whole sample, computed from the SUM of its log returns
total_from_logs = ...
total_from_logs = p.check("total return from log returns", total_from_logs, prices["SPY"].iloc[-1] / prices["SPY"].iloc[0] - 1)

## 2. Moments, fat tails, normality

In [ ]:
moments = pd.DataFrame({t: p.describe(rets[t]) for t in rets}).T
moments

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
r = rets["SPY"]
# ✍️ standardize SPY returns (z = (r - mean) / std) and count days with |z| > 4
z = ...
n_tail = ...
n_tail = p.check("SPY days beyond 4 sigma", n_tail, p.tail_count(r)[0])
print(f"A normal distribution would predict about {p.tail_count(r)[1]:.2f} such days in {len(r)} days.")

In [ ]:
from scipy import stats
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
(osm, osr), (slope, icpt, _) = stats.probplot(rets["SPY"], dist="norm")
axes[0].scatter(osm, osr, s=6, color=p.PALETTE[0]); axes[0].plot(osm, slope * osm + icpt, color="#52514e", lw=1)
axes[0].set(title="QQ plot vs normal: SPY", xlabel="normal quantiles", ylabel="return quantiles")
df, loc, scale = stats.t.fit(rets["SPY"])
x = np.linspace(rets["SPY"].min(), rets["SPY"].max(), 400)
axes[1].hist(rets["SPY"], bins=150, density=True, color="#b7d3f6", label="SPY daily returns")
axes[1].plot(x, stats.norm.pdf(x, rets["SPY"].mean(), rets["SPY"].std()), label="Normal fit")
axes[1].plot(x, stats.t.pdf(x, df, loc, scale), label=f"Student-t fit (df = {df:.1f})")
axes[1].set(title="Distribution of SPY returns", yscale="log", ylim=(1e-2, None)); axes[1].legend()
plt.tight_layout(); plt.show()

## 3. The stylized facts (clinic W1 table)

In [ ]:
rows = {}
for t in rets:
    r = rets[t]
    rows[t] = {"skew": p.describe(r)["skew"], "excess kurtosis": p.describe(r)["excess_kurtosis"],
               "JB p-value": p.describe(r)["jb_pvalue"], ">4σ days": p.tail_count(r)[0],
               ">4σ expected": p.tail_count(r)[1], "t d.f.": stats.t.fit(r)[0],
               "acf(r)": p.autocorr(r), "acf(|r|)": p.autocorr(r.abs()),
               "leverage corr(r, next |r|)": r.corr(r.abs().shift(-1))}
facts = pd.DataFrame(rows).T
facts

In [ ]:
lags = range(1, 21)
acf_r = [rets["SPY"].autocorr(k) for k in lags]
acf_abs = [rets["SPY"].abs().autocorr(k) for k in lags]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
axes[0].bar(lags, acf_r, color=p.PALETTE[0]); axes[0].set_title("Autocorrelation of returns (SPY)")
axes[1].bar(lags, acf_abs, color=p.PALETTE[0]); axes[1].set_title("Autocorrelation of |returns| (SPY)")
for ax in axes: ax.axhline(0, color="#52514e", lw=1); ax.set_xlabel("lag (days)")
plt.tight_layout(); plt.show()

## 4. Aggregational Gaussianity: longer horizons look more normal

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
# ✍️ monthly log returns of SPY = sum of daily log returns in each month (resample("ME")), then excess kurtosis
monthly = ...
kurt_monthly = ...
kurt_monthly = p.check("SPY monthly excess kurtosis", kurt_monthly, stats.kurtosis(rets["SPY"].resample("ME").sum()))
print(f"Excess kurtosis: daily {stats.kurtosis(rets['SPY']):.2f} vs monthly {kurt_monthly:.2f}")

## Questions (clinic W1 deliverable)
Write one paragraph per stylized fact using the table: fat tails, volatility clustering, (almost) no autocorrelation in returns,
leverage effect, aggregational Gaussianity. Which tickers break a pattern, and why might they?